# Movie Rating Predictor v3

In this project, I build a machine learning model to classify whether a movie is highly rated (IMDb ≥ 8) using feature engineering, model comparison, and cross-validation.

In [72]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
df = pd.read_csv("data/movies_data_cleaned.csv")

In [3]:
df.head()

,Series_Title,Released_Year,Certificate,Runtime_Minutes,Genre,Subgenre,Subgenre 1,IMDB_Rating,Meta_score,Director,Star1,Star2,Star3,Star4,No_of_Votes,Gross,Gross_Millions
0,The Shawshank Redemption,1994.0,A,142,Drama,Unknown,Unkown,9.3,80.0,Frank Darabont,Tim Robbins,Morgan Freeman,Bob Gunton,William Sadler,2343110,28341469.0,28.341469
1,The Godfather,1972.0,A,175,Crime,Drama,Unkown,9.2,100.0,Francis Ford Coppola,Marlon Brando,Al Pacino,James Caan,Diane Keaton,1620367,134966411.0,134.966411
2,The Dark Knight,2008.0,UA,152,Action,Crime,Drama,9.0,84.0,Christopher Nolan,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine,2303232,534858444.0,534.858444
3,The Godfather: Part II,1974.0,A,202,Crime,Drama,Unkown,9.0,90.0,Francis Ford Coppola,Al Pacino,Robert De Niro,Robert Duvall,Diane Keaton,1129952,57300000.0,57.300000
4,12 Angry Men,1957.0,U,96,Crime,Drama,Unkown,9.0,96.0,Sidney Lumet,Henry Fonda,Lee J. Cobb,Martin Balsam,John Fiedler,689845,4360000.0,4.360000


In [4]:
df.shape

(1000, 17)

In [5]:
df.columns

Index(['Series_Title', 'Released_Year', 'Certificate', 'Runtime_Minutes',
       'Genre', 'Subgenre', 'Subgenre 1', 'IMDB_Rating', 'Meta_score',
       'Director', 'Star1', 'Star2', 'Star3', 'Star4', 'No_of_Votes', 'Gross',
       'Gross_Millions'],
      dtype='str')

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Series_Title     1000 non-null   str    
 1   Released_Year    999 non-null    float64
 2   Certificate      1000 non-null   str    
 3   Runtime_Minutes  1000 non-null   int64  
 4   Genre            1000 non-null   str    
 5   Subgenre         1000 non-null   str    
 6   Subgenre 1       1000 non-null   str    
 7   IMDB_Rating      1000 non-null   float64
 8   Meta_score       843 non-null    float64
 9   Director         1000 non-null   str    
 10  Star1            1000 non-null   str    
 11  Star2            1000 non-null   str    
 12  Star3            1000 non-null   str    
 13  Star4            1000 non-null   str    
 14  No_of_Votes      1000 non-null   int64  
 15  Gross            831 non-null    float64
 16  Gross_Millions   831 non-null    float64
dtypes: float64(5), int64(2), s

In [7]:
df.isna().sum()

Series_Title         0
Released_Year        1
Certificate          0
Runtime_Minutes      0
Genre                0
Subgenre             0
Subgenre 1           0
IMDB_Rating          0
Meta_score         157
Director             0
Star1                0
Star2                0
Star3                0
Star4                0
No_of_Votes          0
Gross              169
Gross_Millions     169
dtype: int64

## Create Target

In [8]:
df["Target"] = (df["IMDB_Rating"] >= 8.0).astype(int)

In [9]:
df[["Series_Title", "IMDB_Rating", "Target"]].head()

,Series_Title,IMDB_Rating,Target
0,The Shawshank Redemption,9.3,1
1,The Godfather,9.2,1
2,The Dark Knight,9.0,1
3,The Godfather: Part II,9.0,1
4,12 Angry Men,9.0,1


In [10]:
df["Target"].value_counts()

Target
0    537
1    463
Name: count, dtype: int64

In [11]:
df["Target"].value_counts(normalize=True)

Target
0    0.537
1    0.463
Name: proportion, dtype: float64

## Feature Engineering

In [12]:
df["Director"].value_counts()

Director
Alfred Hitchcock       14
Steven Spielberg       13
Hayao Miyazaki         11
Martin Scorsese        10
Akira Kurosawa         10
                       ..
Martin Rosen            1
Wolfgang Reitherman     1
Richard Lester          1
Blake Edwards           1
George Stevens          1
Name: count, Length: 548, dtype: int64

In [15]:
director_counts = df["Director"].value_counts()

In [16]:
common_directors = director_counts[director_counts >= 3].index

In [18]:
df["Director_grouped"] = df["Director"].apply(
    lambda x: x if x in common_directors else "Other")

In [22]:
star_counts = df["Star1"].value_counts()

In [23]:
common_stars = star_counts[star_counts >= 2].index

In [25]:
df["Star1_grouped"] = df["Star1"].apply(
    lambda x: x if x in common_stars else "Other")

In [27]:
df["Director_grouped"].value_counts() 

Director_grouped
Other               550
Alfred Hitchcock     14
Steven Spielberg     13
Hayao Miyazaki       11
Martin Scorsese      10
                   ... 
Jim Jarmusch          3
John Hughes           3
Terrence Malick       3
James Whale           3
Don Siegel            3
Name: count, Length: 98, dtype: int64

In [28]:
df["Star1_grouped"].value_counts()

Star1_grouped
Other                503
Tom Hanks             12
Robert De Niro        11
Al Pacino             10
Clint Eastwood        10
                    ... 
Josh Hartnett          2
Paddy Considine        2
Nicolas Cage           2
Catherine Deneuve      2
Michael B. Jordan      2
Name: count, Length: 158, dtype: int64

In [29]:
df["Log_Votes"] = np.log1p(df["No_of_Votes"])

In [30]:
df[["No_of_Votes", "Log_Votes"]].head()

,No_of_Votes,Log_Votes
0,2343110,14.666990
1,1620367,14.298164
2,2303232,14.649824
3,1129952,13.937687
4,689845,13.444224


## Feature Selection

In [31]:
features = [
    "Runtime_Minutes",
    "Log_Votes",
    "Released_Year",
    "Meta_score",
    "Certificate",
    "Genre",
    "Director_grouped",
    "Star1_grouped"
]

In [32]:
X = df[features]
y = df["Target"]

In [35]:
X.isna().sum()

Runtime_Minutes       0
Log_Votes             0
Released_Year         1
Meta_score          157
Certificate           0
Genre                 0
Director_grouped      0
Star1_grouped         0
dtype: int64

In [36]:
X_clean = X.copy()

In [37]:
numerical_cols = [
    "Runtime_Minutes",
    "Log_Votes",
    "Released_Year",
    "Meta_score"
]

In [38]:
for col in numerical_cols:
    X_clean[col] = X_clean[col].fillna(X_clean[col].median())

In [39]:
categorical_cols = [
    "Certificate",
    "Genre",
    "Director_grouped",
    "Star1_grouped"
]

In [40]:
for col in categorical_cols:
    X_clean[col] = X_clean[col].fillna("Unknown")

In [41]:
X_clean.isna().sum()

Runtime_Minutes     0
Log_Votes           0
Released_Year       0
Meta_score          0
Certificate         0
Genre               0
Director_grouped    0
Star1_grouped       0
dtype: int64

In [42]:
X_encoded = pd.get_dummies(X_clean, drop_first=True)

In [43]:
X_encoded.head()

,Runtime_Minutes,Log_Votes,Released_Year,Meta_score,Certificate_A,Certificate_Approved,Certificate_G,Certificate_GP,Certificate_PG,Certificate_PG-13,...,Star1_grouped_Toshirô Mifune,Star1_grouped_Ulrich Thomsen,Star1_grouped_Uma Thurman,Star1_grouped_Viggo Mortensen,Star1_grouped_Wagner Moura,Star1_grouped_Will Smith,Star1_grouped_Willem Dafoe,Star1_grouped_William Holden,Star1_grouped_Woody Allen,Star1_grouped_Yun-Fat Chow
0,142,14.666990,1994.0,80.0,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,175,14.298164,1972.0,100.0,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,152,14.649824,2008.0,84.0,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,202,13.937687,1974.0,90.0,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,96,13.444224,1957.0,96.0,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [44]:
X_encoded.shape

(1000, 287)

In [45]:
X_encoded.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Columns: 287 entries, Runtime_Minutes to Star1_grouped_Yun-Fat Chow
dtypes: bool(283), float64(3), int64(1)
memory usage: 307.7 KB


## Train/Test Split

In [46]:
X_train, X_test, y_train, y_test = train_test_split(X_encoded,y,test_size=0.2,random_state=42)

In [47]:
print(X_train.shape)
print(X_test.shape)

(800, 287)
(200, 287)


In [48]:
print(y_train.shape)
print(y_test.shape)

(800,)
(200,)


In [49]:
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

Target
0    0.535
1    0.465
Name: proportion, dtype: float64
Target
0    0.545
1    0.455
Name: proportion, dtype: float64


## logistic Regression

In [50]:
log_model = LogisticRegression(max_iter=1000)

In [51]:
log_model.fit(X_train, y_train)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [52]:
y_pred_log = log_model.predict(X_test)

In [53]:
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_log))

Logistic Regression Accuracy: 0.685


In [54]:
print(classification_report(y_test, y_pred_log))

              precision    recall  f1-score   support

           0       0.71      0.72      0.71       109
           1       0.66      0.65      0.65        91

    accuracy                           0.69       200
   macro avg       0.68      0.68      0.68       200
weighted avg       0.68      0.69      0.68       200



In [55]:
print(confusion_matrix(y_test, y_pred_log))

[[78 31]
 [32 59]]


In [56]:
cv_scores_log = cross_val_score(log_model, X_encoded, y, cv=5, scoring="accuracy")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:

In [57]:
cv_scores_log

array([0.69 , 0.73 , 0.6  , 0.655, 0.605])

In [58]:
cv_scores_log.mean()

np.float64(0.6559999999999999)

In [59]:
print("Cross-validation scores:", cv_scores_log)
print("Mean CV accuracy:", cv_scores_log.mean())

Cross-validation scores: [0.69  0.73  0.6   0.655 0.605]
Mean CV accuracy: 0.6559999999999999


## Random Forest

In [60]:
forest_model = RandomForestClassifier(random_state=42)

In [61]:
forest_model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [62]:
y_pred_forest = forest_model.predict(X_test)

In [63]:
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_forest))

Random Forest Accuracy: 0.685


In [64]:
print(classification_report(y_test, y_pred_forest))

              precision    recall  f1-score   support

           0       0.69      0.77      0.73       109
           1       0.68      0.58      0.63        91

    accuracy                           0.69       200
   macro avg       0.68      0.68      0.68       200
weighted avg       0.68      0.69      0.68       200



In [65]:
print(confusion_matrix(y_test, y_pred_forest))

[[84 25]
 [38 53]]


In [66]:
cv_scores_forest = cross_val_score(forest_model, X_encoded, y, cv=5, scoring="accuracy")

In [67]:
print("Random Forest CV scores:", cv_scores_forest)
print("Random Forest Mean CV accuracy:", cv_scores_forest.mean())

Random Forest CV scores: [0.745 0.78  0.665 0.71  0.63 ]
Random Forest Mean CV accuracy: 0.706


In [68]:
log_test_acc = accuracy_score(y_test, y_pred_log)
log_cv_mean = cv_scores_log.mean()

In [69]:
forest_test_acc = accuracy_score(y_test, y_pred_forest)
forest_cv_mean = cv_scores_forest.mean()

## Model Comparison

In [70]:
results_v3 = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Test Accuracy": [log_test_acc, forest_test_acc],
    "CV Accuracy (Mean)": [log_cv_mean, forest_cv_mean]
})

results_v3

,Model,Test Accuracy,CV Accuracy (Mean)
0,Logistic Regression,0.685,0.656
1,Random Forest,0.685,0.706


In [71]:
results_v3.sort_values(by="Test Accuracy", ascending=False)

,Model,Test Accuracy,CV Accuracy (Mean)
0,Logistic Regression,0.685,0.656
1,Random Forest,0.685,0.706


## Final Conclusion

In this project, I built a model to predict whether a movie is highly rated.

I improved the features and compared Logistic Regression and Random Forest using cross-validation. Random Forest achieved better performance.

This project shows the importance of feature engineering and reliable evaluation in machine learning.